# Adding and changing modules

A simulated human in embodied-human is one **module** per **category**
(eyes, attention, memory, touch, vestibular, ...). Every category has an
abstract interface, a cited baseline and sometimes alternatives. This
tutorial shows how to:

1. inspect the baseline and where each parameter comes from;
2. change a module's parameters;
3. swap in another registered implementation;
4. change the agent that uses the modules;
5. write and register a new module;
6. see what the change does to integrated behaviour on the benchmark;
7. test the new module;
8. add a human reference and a benchmark of your own.

Everything here runs on the straight-road simulation, so it needs neither
MyoSuite nor a GPU. Install the package from the repository root with
`pip install -e .`; the whole notebook runs in about a minute.

In [1]:
import embodied_human as eh

for category, names in eh.available().items():
    print(f"{category:15} {', '.join(names)}")

eyes            main_sequence
vision          foveated
objects         eccentricity_noise
attention       seev, uncertainty_sampler
memory          act_r
proprioception  spindle
motor           ffitts_min_jerk
estimator       lane_kalman
hearing         psychoacoustic
touch           vibrotactile
vestibular      iso_2631, subjective_vertical
response        twin
fatigue         three_compartment
affect          appraisal
risk            risk_field
trust           decision_field


## 1. The baseline and its sources

`build_human` creates one module per category from a `HumanConfig`. The
default config is the cited baseline. Each module keeps its parameters in
a frozen dataclass, and every field carries a `Source`: whether the value
was **measured** in humans, set by a **standard**, **fitted** to human
data, **derived** from other values, or **assumed** (a modelling choice to
fit or replace). `provenance()` lists them.

In [2]:
human = eh.build_human(seed=0)
print(type(human.attention).__name__)
for row in human.provenance():
    if row["category"] == "attention":
        name, value = row["parameter"], row["value"]
        print(f"{name:38} {value!s:6} {row['kind']:8} {row['citation']}")

UncertaintySampler
SamplerParameters.return_sd            0.11   fitted   Kujala & Saariluoma (2011), Table 2; fitted here by ABC
SamplerParameters.threshold_spread     1.0    fitted   Kujala & Saariluoma (2011), Table 2; fitted here by ABC
SamplerParameters.leave_sd             0.15   assumed  model choice
SamplerParameters.minimum_road_dwell   0.3    assumed  model choice
SamplerParameters.maximum_glance       4.0    assumed  model choice


## 2. Change a module's parameters

`HumanConfig.replace(category, name, **params)` returns a new config with
one category swapped; parameters you do not name keep their defaults.
Here the attention module tolerates more uncertainty about the lane before
looking back at the road. `simulate_glances` runs the attentive driver
doing a visual search task (SuRT) on a straight road; its log explains
every glance.

In [3]:
from embodied_human.agents.road import simulate_glances

KMH = 1 / 3.6


def glances(config, seed=0, speed_kmh=50.0, seconds=120.0):
    human = eh.build_human(config, seed=seed)
    log = simulate_glances(human, speed=speed_kmh * KMH, seconds=seconds, seed=seed)
    return log.glance_metrics()


baseline = eh.HumanConfig()
tolerant = baseline.replace("attention", "uncertainty_sampler", return_sd=0.2)
for name, config in (("baseline", baseline), ("return_sd=0.2", tolerant)):
    m = glances(config)
    print(
        f"{name:14} mean glance {m['mean_offroad_glance_s']:.2f} s, "
        f"over 2 s {100 * m['long_glance_share']:.1f}%, "
        f"returns {m['returns_by_reason']}"
    )

baseline       mean glance 1.08 s, over 2 s 4.7%, returns {'uncertainty': 96}


return_sd=0.2  mean glance 1.47 s, over 2 s 17.9%, returns {'uncertainty': 73, 'maximum_glance': 1}


## 3. Swap in another implementation

Implementations of one category share an interface, so they are
interchangeable. The vestibular category has the subjective vertical
conflict model and the ISO 2631-1 motion sickness dose; both turn head
motion into the percentage of people expected to vomit.

In [4]:
import math

import numpy as np

dt = 0.1
t = np.arange(0, 1800, dt)  # 30 minutes
motion = np.zeros((len(t), 3))
motion[:, 2] = 1.0 * np.sin(2 * math.pi * 0.2 * t)  # vertical, 0.2 Hz, 1 m/s^2

for name in ("subjective_vertical", "iso_2631"):
    config = eh.HumanConfig().replace("vestibular", name)
    msi = eh.build_human(config, seed=0).vestibular.simulate(dt, motion)
    print(f"{name:20} {msi[-1]:5.1f}% after 30 min")

subjective_vertical   11.3% after 30 min
iso_2631               9.9% after 30 min


## 4. Change the agent instead of a module

Agents such as the attentive driver combine the modules with task
geometry and their own parameters (`DriverAgentParameters`), each with a
source too. For example, the time the hand needs to let go of the task
after a take-over request depends on the task: it is fitted to a
touchscreen task and assumed to be zero for a rotary knob.

In [5]:
from dataclasses import replace

from embodied_human.agents.driver import DriverAgentParameters
from embodied_human.agents.road import simulate_takeover

for release in (DriverAgentParameters().hand_release, 0.0):
    params = replace(DriverAgentParameters(), hand_release=release)
    log = simulate_takeover(
        eh.build_human(seed=1), speed=120 * KMH, seed=1, params=params
    )
    parts = ", ".join(f"{k} {v:.2f}" for k, v in log.takeover_components().items())
    print(f"hand_release {release:.2f} s: {parts}")

hand_release 0.59 s: response_s 0.28, gaze_s 0.45, hand_s 1.29, eyes_s 0.54, engaged_s 1.30


hand_release 0.00 s: response_s 0.28, gaze_s 0.45, hand_s 0.69, eyes_s 0.54, engaged_s 1.30


## 5. Write a new module

A new module subclasses the category's interface, keeps its parameters in
a frozen dataclass with sources, and registers itself under a name. The
attention interface has one method, `select(time, current, demands,
uncertainty)`, which returns the area to look at next: `"road"` or one of
the `demands` (areas that want attention, such as the task display).

As an example, a **fixed-interval sampler** looks away for a fixed time
and back at the road for a fixed time, whatever the driving situation.
The baseline instead returns when its belief about the lane grows
uncertain. Setting `reason` lets the driver's log explain each return to
the road.

In [6]:
from collections.abc import Mapping
from dataclasses import dataclass
from typing import ClassVar

from embodied_human.attention import ROAD, Attention


@dataclass(frozen=True)
class FixedIntervalParameters(eh.Parameters):
    """Fixed glance and road times.

    Args:
        glance: Time looking away from the road, s.
        road_dwell: Time back on the road before looking away again, s.
    """

    glance: float = 1.0
    road_dwell: float = 0.6

    SOURCES: ClassVar = {
        "glance": eh.Source(
            "model choice", None, "assumed", "about the mean in-vehicle glance"
        ),
        "road_dwell": eh.Source(
            "Zeeb, Buchner & Schrauf (2015), Section 4.1",
            "https://doi.org/10.1016/j.aap.2015.02.023",
            "measured",
            "mean road glance during automation with a visual task, 0.6 s",
        ),
    }


class FixedIntervalSampler(Attention):
    """Look away for ``glance`` seconds, then at the road for ``road_dwell``."""

    def __init__(self, **params: float) -> None:
        super().__init__()
        self.params = FixedIntervalParameters(**params)
        self.reset()

    def reset(self, rng=None) -> None:
        super().reset(rng)
        self._current, self._since, self.reason = ROAD, 0.0, ""

    def select(
        self,
        time: float,
        current: str,
        demands: Mapping[str, float],
        uncertainty: float,
    ) -> str:
        if current != self._current:
            self._current, self._since = current, time
        dwell = time - self._since
        if current != ROAD:
            if dwell < self.params.glance and current in demands:
                return current
            self.reason = "interval" if current in demands else "task_done"
            return ROAD
        if demands and dwell >= self.params.road_dwell:
            return max(demands, key=lambda name: demands[name])
        return ROAD


# Registering a name twice raises an error, so guard re-runs of this cell.
if "fixed_interval" not in eh.available("attention")["attention"]:
    eh.register("attention", "fixed_interval")(FixedIntervalSampler)
print(eh.available("attention"))

{'attention': ['fixed_interval', 'seev', 'uncertainty_sampler']}


The new module is used like any other, and its parameters and sources
appear in the provenance report:

In [7]:
fixed = eh.HumanConfig().replace("attention", "fixed_interval")
human = eh.build_human(fixed, seed=0)
for row in human.provenance():
    if row["category"] == "attention":
        name, value = row["parameter"], row["value"]
        print(f"{name:36} {value!s:5} {row['kind']:8} {row['citation']}")
m = glances(fixed)
glance, returns = m["mean_offroad_glance_s"], m["returns_by_reason"]
print(f"mean glance {glance:.2f} s, returns {returns}")

FixedIntervalParameters.glance       1.0   assumed  model choice
FixedIntervalParameters.road_dwell   0.6   measured Zeeb, Buchner & Schrauf (2015), Section 4.1


mean glance 1.14 s, returns {'interval': 71}


## 6. What changes in integrated behaviour?

The point of a shared benchmark is to change one module, keep everything
else fixed, and see what the change does. Two glance benchmarks show the
trade-off:

- `driving.glance_population` simulates 20 participants and compares mean
  glance duration, within-person spread and long glances with Kujala and
  Saariluoma (2011).
- `driving.glance_speed` asks how glance duration changes with speed.
  Drivers who may close their eyes at will look away for a roughly
  constant distance (Senders et al., 1967), but in-car search glances do
  not change with speed (Jokinen et al., 2021).

In [8]:
from embodied_human.bench.benchmarks import run
from embodied_human.bench.references import REFERENCES

names = ["driving.glance_population", "driving.glance_speed"]
for label, config in (("baseline", baseline), ("fixed_interval", fixed)):
    print(label)
    for result in run(eh.build_human(config, seed=0), names):
        for key, value in result.metrics.items():
            ref = REFERENCES[key]
            mark = "pass" if result.scores[key] == 1.0 else "fail"
            unit = "" if ref.unit == "1" else f" {ref.unit}"
            human_range = f"{ref.low:g} to {ref.high:g}{unit}"
            print(f"  {key:38} {value:7.3f}  human {human_range}: {mark}")

baseline


  driving.human_mean_glance                1.132  human 0.8 to 1.2 s: pass
  driving.human_glance_sd                  0.582  human 0.48 to 0.66 s: pass
  driving.human_long_glance_share          0.079  human 0.03 to 0.13 fraction: pass
  driving.mean_offroad_glance              1.132  human 0 to 2 s: pass
  driving.long_glance_share                0.079  human 0 to 0.15 fraction: pass
  driving.occlusion_speed_elasticity      -0.453  human -0.72 to -0.53: fail
  driving.incar_glance_speed_elasticity   -0.453  human -0.15 to 0.15: fail
fixed_interval


  driving.human_mean_glance                1.120  human 0.8 to 1.2 s: pass
  driving.human_glance_sd                  0.077  human 0.48 to 0.66 s: fail
  driving.human_long_glance_share          0.000  human 0.03 to 0.13 fraction: fail
  driving.mean_offroad_glance              1.120  human 0 to 2 s: pass
  driving.long_glance_share                0.000  human 0 to 0.15 fraction: pass
  driving.occlusion_speed_elasticity       0.000  human -0.72 to -0.53: fail
  driving.incar_glance_speed_elasticity    0.000  human -0.15 to 0.15: pass


The fixed-interval sampler ignores speed, as in-car search did, but its
glances barely vary within a person, unlike human glances. The
uncertainty-based baseline matches the population data but sits between
the two speed results. Neither passes every check: that is what the
benchmark is for.

From the command line, any registered implementation can replace the
baseline. Put the module in an importable file (say `my_attention.py`)
and name it with `--import`:

```bash
python -m embodied_human.bench --import my_attention --replace attention=fixed_interval --out fixed.json
python -m embodied_human.bench --replace attention=uncertainty_sampler:return_sd=0.2 --only driving.glance_population
```

## 7. Test the module

A module should be checked against its own definition before it is
benchmarked. In the repository these checks live in `tests/` and run with
`python -m pytest tests -q`.

In [9]:
def test_fixed_interval_glances_do_not_depend_on_speed():
    durations = []
    for speed in (40.0, 120.0):
        human = eh.build_human(fixed, seed=0)
        log = simulate_glances(human, speed=speed * KMH, seconds=60, seed=0)
        durations.append(np.mean(log.glance_durations()))
    # The glance time plus the saccades to and from the display.
    assert abs(durations[0] - durations[1]) < 0.05
    assert 1.0 < durations[0] < 1.3


test_fixed_interval_glances_do_not_depend_on_speed()
print("ok")

ok


## 8. Add a human reference and a benchmark

A benchmark is a function of a `Human` that returns a `Result`, scored
against `Reference` values: a published aggregate with an acceptance
range, its citation and the conditions. The range is a published range
where studies disagree, a confidence interval of the mean, or a stated
tolerance. Say in the conditions which data, if any, the model was
fitted to.

In [10]:
from embodied_human.bench.benchmarks import BENCHMARKS, Result, benchmark
from embodied_human.bench.references import Reference

REFERENCES["tutorial.automation_glance"] = Reference(
    "tutorial.automation_glance",
    12.6,
    7.2,
    12.6,
    "s",
    "Zeeb, Buchner & Schrauf (2015), Section 3.1 and 4.1",
    "https://doi.org/10.1016/j.aap.2015.02.023",
    "single glance at a visual task during automation; range from the "
    "median (7.2 s) to the mean (12.6 s)",
)

if "tutorial.automation_glance" not in BENCHMARKS:

    @benchmark("tutorial.automation_glance")
    def automation_glance(human):
        result = Result("tutorial.automation_glance", ("attention",))
        log = simulate_takeover(
            eh.build_human(human.config, seed=0),
            speed=120 * KMH,
            request_at=60,
            seconds=60,
        )
        mean = float(np.mean(log.glance_durations()))
        result.score_against("tutorial.automation_glance", mean)
        return result


for label, config in (("baseline", baseline), ("fixed_interval", fixed)):
    result = BENCHMARKS["tutorial.automation_glance"](eh.build_human(config, seed=0))
    print(f"{label:15} {result.metrics}  score {result.score:.2f}")

baseline        {'tutorial.automation_glance': 0.6817543859649545}  score 0.00


fixed_interval  {'tutorial.automation_glance': 1.1322222222222953}  score 0.00


Neither attention module knows that automation is driving, so both keep
glancing as in manual driving. The packaged benchmark
`driving.automation_monitoring` records this gap with Zeeb et al.'s full
data.

## Contributing a module

1. Put the implementation in the category's file under
   `src/embodied_human/` (or in a new file that `core._load_builtins`
   imports).
2. Give every parameter a `Source`. Label fitted values with what they
   were fitted to, and keep the fitting script in `examples/`.
3. Add tests of its equations to `tests/`.
4. Run the benchmark with `--replace`, and report the results, failures
   included, in `VALIDATION.md`.
5. Regenerate `docs/parameters.md` with
   `python examples/parameter_table.py`.